In [6]:
# ============================================================
# 🎬 MOVIE MATCHMAKER
# Optimized Movie Recommendation System
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import html as _html
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as W

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display, clear_output


# 2. CONFIGURATION
DATA_PATH = "/content/movies_10k.csv"

# Fallback if running outside Google Colab
if not os.path.exists(DATA_PATH):
    DATA_PATH = "movies_1M.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Movie dataset not found.\nExpected file: {DATA_PATH}"
    )

MODEL_NAME = "all-MiniLM-L6-v2"

# Embedding configuration
BATCH_SIZE = 32
EMBEDDING_DTYPE = np.float32

# Recommendation configuration
POOL_SIZE = 200

# Cache files
CACHE_DIR = os.path.dirname(DATA_PATH) or "."
EMB_CACHE = os.path.join(CACHE_DIR, "movie_embeddings.npy")
META_CACHE = os.path.join(CACHE_DIR, "movie_embeddings_meta.npz")
print("Loading movie dataset...")

df = pd.read_csv(
    DATA_PATH,
    engine="python",
    on_bad_lines="warn"
)

df = df.reset_index(drop=True)

print(f"Loaded {len(df):,} movies.")


# CLEAN DATA

required_columns = [
    "Title",
    "Genre",
    "Overview",
    "Keywords",
    "Rating"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


df["Title"] = df["Title"].fillna("").astype(str)
df["Genre"] = df["Genre"].fillna("").astype(str)
df["Overview"] = df["Overview"].fillna("").astype(str)
df["Keywords"] = df["Keywords"].fillna("").astype(str)

df["Rating"] = pd.to_numeric(
    df["Rating"],
    errors="coerce"
)


# CREATE COMBINED TEXT
# Combining the fields used by the original recommendation system

df["Combine Text"] = (
    df["Genre"]
    + " "
    + df["Overview"]
    + " "
    + df["Keywords"]
).str.strip()



# 6. OPTIONAL TF-IDF
# Kept because it was part of the original project.
# It can be removed if you only want Sentence-BERT.

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=100000
)

vectors = vectorizer.fit_transform(
    df["Combine Text"]
)

print(
    f"TF-IDF matrix created: {vectors.shape}"
)



# LOAD SENTENCE-BERT


print("Loading Sentence-BERT model...")

model = SentenceTransformer(MODEL_NAME)

print("Sentence-BERT model loaded successfully.")



# EMBEDDING CACHE


def create_embeddings():
    """
    Create movie embeddings in batches.

    This avoids sending the complete movie dataset through
    Sentence-BERT as one enormous operation.
    """

    texts = df["Combine Text"].tolist()

    print(
        f"Creating embeddings for {len(texts):,} movies..."
    )

    print(
        f"Batch size: {BATCH_SIZE}"
    )

    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embeddings = embeddings.astype(
        EMBEDDING_DTYPE,
        copy=False
    )

    np.save(
        EMB_CACHE,
        embeddings
    )

    np.savez(
        META_CACHE,
        row_count=len(df),
        embedding_dim=embeddings.shape[1]
    )

    print(
        f"Embeddings saved to: {EMB_CACHE}"
    )

    return embeddings


def load_cached_embeddings():
    """
    Load cached embeddings only when they match
    the current dataset.
    """

    if not os.path.exists(EMB_CACHE):
        return None

    try:

        embeddings = np.load(
            EMB_CACHE,
            mmap_mode="r"
        )

        if embeddings.ndim != 2:
            print("Invalid embedding cache.")
            return None

        if embeddings.shape[0] != len(df):
            print(
                "Embedding cache does not match dataset size."
            )
            return None

        print(
            f"Loaded cached embeddings: {embeddings.shape}"
        )

        return np.asarray(
            embeddings,
            dtype=EMBEDDING_DTYPE
        )

    except Exception as e:

        print(
            f"Could not load embedding cache: {e}"
        )

        return None


# LOAD OR CREATE EMBEDDINGS


movie_embeddings = load_cached_embeddings()

if movie_embeddings is None:

    movie_embeddings = create_embeddings()

else:

    print(
        "Using existing movie embeddings."
    )


print(
    f"Final embedding shape: {movie_embeddings.shape}"
)


 # COLUMN CONFIGURATION


title_column = "Title"
rating_column = "Rating"
genre_column = "Genre"

# FAST SIMILARITY FUNCTION


def calculate_similarity(query_embedding):
    """
    Because both movie embeddings and query embeddings are
    normalized, cosine similarity becomes a simple matrix
    multiplication.
    """

    query_embedding = np.asarray(
        query_embedding,
        dtype=EMBEDDING_DTYPE
    )

    return np.dot(
        movie_embeddings,
        query_embedding.T
    ).ravel()



#RECOMMENDATION ENGINE

def recommend_movies(
    user_input=None,
    number_of_movies=10,
    min_rating=0.0,
    genre=None,
    sort_by="match",
    hide_duplicates=True,
    query_embedding=None,
    exclude_title=None
):

    """
    Return recommended movies.

    sort_by:
        match  -> closest semantic matches
        rating -> highest rated among good matches
        blend  -> similarity + rating
    """

    # --------------------------------------------------------
    # Create query embedding
    # --------------------------------------------------------

    if query_embedding is None:

        if not user_input:
            return df.iloc[0:0].copy()

        query_embedding = model.encode(
            [user_input],
            convert_to_numpy=True,
            normalize_embeddings=True
        )

    # --------------------------------------------------------
    # Similarity
    # --------------------------------------------------------

    similarity = calculate_similarity(
        query_embedding
    )

    # --------------------------------------------------------
    # Filters
    # --------------------------------------------------------

    keep = np.ones(
        len(df),
        dtype=bool
    )

    if min_rating > 0:

        keep &= (
            df[rating_column]
            .fillna(-1)
            .to_numpy()
            >= min_rating
        )

    if genre:

        keep &= (
            df[genre_column]
            .str.contains(
                genre,
                case=False,
                regex=False,
                na=False
            )
            .to_numpy()
        )

    if exclude_title:

        keep &= (
            df[title_column]
            != exclude_title
        ).to_numpy()

    candidates = np.flatnonzero(
        keep
    )

    if len(candidates) == 0:

        return df.iloc[0:0].assign(
            similarity=[],
            similarity_percentage=[],
            Rank=[]
        )

    # --------------------------------------------------------
    # Get strongest matches
    # --------------------------------------------------------

    # Only sort candidates rather than the entire dataframe.
    order = candidates[
        np.argsort(
            -similarity[candidates]
        )
    ]

    pool_size = max(
        POOL_SIZE,
        number_of_movies * 10
    )

    pool = order[:pool_size]

    # --------------------------------------------------------
    # Remove duplicate titles
    # --------------------------------------------------------

    if hide_duplicates:

        seen = set()
        unique = []

        for idx in pool:

            title = df.at[
                idx,
                title_column
            ]

            if title not in seen:

                seen.add(title)
                unique.append(idx)

        pool = np.asarray(
            unique,
            dtype=np.int64
        )

    # --------------------------------------------------------
    # Ranking
    # --------------------------------------------------------

    sim = similarity[pool]

    if sort_by == "rating":

        ratings = (
            df.iloc[pool][rating_column]
            .fillna(-1)
            .to_numpy()
        )

        order = np.lexsort(
            (-sim, -ratings)
        )

        pool = pool[order]

    elif sort_by == "blend":

        ratings = (
            df.iloc[pool][rating_column]
            .fillna(
                df[rating_column].median()
            )
            .to_numpy()
        )

        ratings = ratings / 10.0

        sim_min = sim.min()
        sim_max = sim.max()

        if sim_max > sim_min:

            sim_norm = (
                (sim - sim_min)
                / (sim_max - sim_min)
            )

        else:

            sim_norm = np.ones_like(sim)

        score = (
            0.65 * sim_norm
            + 0.35 * ratings
        )

        pool = pool[
            np.argsort(-score)
        ]


    top = pool[
        :number_of_movies
    ]

    results = df.iloc[top].copy()

    results["similarity"] = similarity[top]

    results["similarity_percentage"] = (
        results["similarity"] * 100
    )

    results["Rank"] = np.arange(
        1,
        len(results) + 1
    )

    return results

def genre_counts(frame):

    counts = {}

    for genres in frame[
        genre_column
    ].astype(str):

        for genre_name in (
            genres
            .replace("|", ",")
            .replace(";", ",")
            .split(",")
        ):

            genre_name = genre_name.strip()

            if genre_name:

                counts[genre_name] = (
                    counts.get(
                        genre_name,
                        0
                    ) + 1
                )

    if not counts:

        return pd.DataFrame(
            columns=["Genre", "Count"]
        )

    return (
        pd.DataFrame(
            list(counts.items()),
            columns=["Genre", "Count"]
        )
        .sort_values(
            "Count",
            ascending=False
        )
    )


print("Recommendation engine ready.")

CSS = """
<style>

.mv-head {
    background:#17313A;
    color:#FBF3E2;
    padding:22px 26px;
    border-radius:14px;
    margin-bottom:6px
}

.mv-head h1 {
    font:700 30px Georgia,serif;
    margin:0 0 6px 0;
    color:#FBF3E2
}

.mv-head p {
    font:15px system-ui,sans-serif;
    margin:0;
    color:#CFE3E1
}

.mv-empty {
    font:16px system-ui,sans-serif;
    color:#555;
    padding:28px;
    text-align:center
}

.mv-note {
    font:600 15px system-ui,sans-serif;
    color:#17313A;
    margin:6px 2px 10px
}

.mv-ticket {
    display:flex;
    background:#FBF3E2;
    border:1px solid #E3D3B0;
    border-radius:14px;
    margin:0 0 12px 0;
    overflow:hidden;
    color:#26323A;
    font-family:system-ui,sans-serif
}

.mv-stub {
    flex:0 0 74px;
    background:#F2A93B;
    display:flex;
    align-items:center;
    justify-content:center;
    font:700 30px Georgia,serif;
    color:#17313A;
    border-right:2px dashed #17313A
}

.mv-body {
    flex:1;
    padding:12px 16px;
    min-width:0
}

.mv-title {
    font:700 19px Georgia,serif;
    color:#17313A;
    margin:0 0 4px
}

.mv-meta {
    font-size:13px;
    color:#5B5140;
    margin-bottom:6px
}

.mv-stars {
    color:#B8352E;
    letter-spacing:1px;
    font-size:15px
}

.mv-pill {
    display:inline-block;
    background:#2F6F73;
    color:#fff;
    border-radius:99px;
    padding:1px 10px;
    font-size:12px;
    margin:0 4px 4px 0
}

.mv-bar {
    background:#E8DCC0;
    border-radius:99px;
    height:9px;
    margin:6px 0 2px;
    overflow:hidden
}

.mv-fill {
    height:9px;
    border-radius:99px;
    background:linear-gradient(90deg,#2F6F73,#F2A93B)
}

.mv-pct {
    font-size:12px;
    color:#5B5140
}

.mv-over {
    font-size:13.5px;
    line-height:1.45;
    color:#3B3B3B;
    margin-top:6px
}

</style>
"""


def esc(x):

    return _html.escape(
        str(x)
    )


def stars(r):

    if pd.isna(r):

        return "no rating yet"

    filled = int(
        round(r / 2)
    )

    return (
        f"<span class='mv-stars'>"
        f"{'★' * filled}"
        f"{'☆' * (5 - filled)}"
        f"</span> "
        f"{r:.1f} out of 10"
    )


def ticket(row):

    rank = int(
        row["Rank"]
    )

    badge = {
        1: "🥇",
        2: "🥈",
        3: "🥉"
    }.get(
        rank,
        rank
    )

    genres = (
        str(row[genre_column])
        .replace("|", ",")
        .replace(";", ",")
        .split(",")
    )

    pills = "".join(
        f"<span class='mv-pill'>"
        f"{esc(g.strip())}"
        f"</span>"
        for g in genres
        if g.strip()
    )

    pct = float(
        row["similarity_percentage"]
    )

    overview = str(
        row.get(
            "Overview",
            ""
        )
    )

    overview = (
        overview[:190] + "…"
        if len(overview) > 190
        else overview
    )

    overview_html = (
        f"<div class='mv-over'>"
        f"{esc(overview)}"
        f"</div>"
        if overview
        else ""
    )

    return f"""
    <div class='mv-ticket'>

        <div class='mv-stub'>
            {badge}
        </div>

        <div class='mv-body'>

            <div class='mv-title'>
                {esc(row[title_column])}
            </div>

            <div class='mv-meta'>
                {stars(row[rating_column])}
            </div>

            <div>
                {pills}
            </div>

            <div class='mv-bar'>
                <div
                    class='mv-fill'
                    style='width:{max(2, min(100, pct)):.0f}%'>
                </div>
            </div>

            <div class='mv-pct'>
                {pct:.1f}% match
            </div>

            {overview_html}

        </div>

    </div>
    """

all_genres = sorted({

    genre_name.strip()

    for genre_list in df[
        genre_column
    ].astype(str)

    for genre_name in (
        genre_list
        .replace("|", ",")
        .replace(";", ",")
        .split(",")
    )

    if genre_name.strip()
})



header = W.HTML(
    CSS
    + f"""
    <div class='mv-head'>

        <h1>
            🎬 Movie Matchmaker
        </h1>

        <p>
            Say what you feel like watching.
            The AI reads the plot, genre and keywords
            of {len(df):,} movies and finds the closest matches.
        </p>

    </div>
    """
)


query = W.Text(
    placeholder=(
        "Describe a movie, a mood or a plot..."
    ),
    layout=W.Layout(
        width="100%",
        height="42px"
    )
)


search_btn = W.Button(
    description="Find movies",
    icon="search",
    button_style="warning",
    layout=W.Layout(
        width="160px",
        height="42px"
    )
)


surprise_btn = W.Button(
    description="Surprise me",
    icon="random",
    layout=W.Layout(
        width="150px",
        height="42px"
    )
)


PROMPT_CHIPS = [

    (
        "🚀 Space adventure",
        "A science fiction movie about space exploration and astronauts"
    ),

    (
        "😂 Feel-good comedy",
        "A light, funny comedy that makes me laugh with friends and family"
    ),

    (
        "🕵️ Mind-bending thriller",
        "A tense psychological thriller with twists and a shocking ending"
    ),

    (
        "💔 Tearjerker",
        "An emotional romantic drama about love, loss and second chances"
    ),

    (
        "👻 Spooky night",
        "A scary horror movie with a haunted house and a supernatural mystery"
    ),

    (
        "🧸 Family fun",
        "An animated adventure the whole family will enjoy"
    )

]


SURPRISES = [
    prompt
    for _, prompt in PROMPT_CHIPS
] + [

    "A heist crew plans one last impossible job",

    "A small-town detective solves a strange disappearance",

    "A documentary about nature and wild animals",

    "A war epic about courage and sacrifice",

    "A superhero discovers their powers for the first time",

    "A quirky story about friendship and growing up"

]


chip_buttons = [

    W.Button(
        description=label,
        tooltip=prompt,
        layout=W.Layout(
            width="auto",
            margin="2px 4px 2px 0"
        )
    )

    for label, prompt in PROMPT_CHIPS

]


n_slider = W.IntSlider(
    value=10,
    min=3,
    max=20,
    description="How many",
    continuous_update=False,
    style={
        "description_width": "80px"
    },
    layout=W.Layout(
        width="300px"
    )
)


rating_slider = W.FloatSlider(
    value=0,
    min=0,
    max=9,
    step=0.5,
    description="Min rating",
    continuous_update=False,
    style={
        "description_width": "80px"
    },
    layout=W.Layout(
        width="300px"
    )
)


genre_dd = W.Dropdown(
    options=[
        "Any genre"
    ] + all_genres,
    description="Genre",
    style={
        "description_width": "80px"
    },
    layout=W.Layout(
        width="300px"
    )
)


sort_toggle = W.ToggleButtons(
    options=[
        ("Best match", "match"),
        ("Top rated", "rating"),
        ("Balanced", "blend")
    ],
    value="match"
)


dedupe = W.Checkbox(
    value=True,
    description="Hide repeated titles",
    indent=False
)



similar_dd = W.Dropdown(
    options=[],
    description="Pick one",
    disabled=True,
    style={
        "description_width": "70px"
    },
    layout=W.Layout(
        width="420px"
    )
)


similar_btn = W.Button(
    description="More like this",
    icon="clone",
    disabled=True,
    layout=W.Layout(
        width="160px"
    )
)


status = W.HTML("")

results_html = W.HTML(
    "<div class='mv-empty'>"
    "Pick a mood above, or describe what you feel like "
    "watching and press <b>Find movies</b>."
    "</div>"
)

charts_out = W.Output()

save_btn = W.Button(
    description="Save results as CSV",
    icon="download",
    disabled=True,
    layout=W.Layout(
        width="200px"
    )
)

save_msg = W.HTML("")


about = W.HTML(
    """
    <div style='font:14px/1.6 system-ui,sans-serif;
                padding:8px 4px;color:#333'>

        <b>How it works</b><br>

        Every movie's genre, overview and keywords are
        converted into an embedding using Sentence-BERT.

        Your description receives the same treatment.

        The system then compares the embeddings to find
        semantically similar movies.

        <br><br>

        <b>Technologies used:</b>

        Natural Language Processing,
        Sentence-BERT,
        sentence embeddings,
        cosine similarity,
        recommendation systems,
        data analytics and visualization.

    </div>
    """
)

state = {
    "results": None,
    "charts_stale": False
}


def show_results(
    results,
    note
):

    state["results"] = results

    state["charts_stale"] = True

    if len(results) == 0:

        results_html.value = (
            "<div class='mv-empty'>"
            "No movies match those filters. "
            "Try a lower minimum rating or choose "
            "<b>Any genre</b>."
            "</div>"
        )

        similar_dd.options = []

        similar_dd.disabled = True

        similar_btn.disabled = True

        save_btn.disabled = True

        return

    results_html.value = (
        f"<div class='mv-note'>"
        f"{esc(note)}"
        f"</div>"
        + "".join(
            ticket(row)
            for _, row in results.iterrows()
        )
    )

    similar_dd.options = [
        (
            f"{int(row['Rank'])}. "
            f"{row[title_column]}",
            int(index)
        )

        for index, row
        in results.iterrows()
    ]

    similar_dd.disabled = False
    similar_btn.disabled = False
    save_btn.disabled = False
    save_msg.value = ""

    if tabs.selected_index == 1:

        draw_charts()

def current_filters():

    return {

        "number_of_movies":
            n_slider.value,

        "min_rating":
            rating_slider.value,

        "genre":
            None
            if genre_dd.value == "Any genre"
            else genre_dd.value,

        "sort_by":
            sort_toggle.value,

        "hide_duplicates":
            dedupe.value
    }


def busy(
    on,
    text=""
):

    for button in (
        [search_btn, surprise_btn]
        + chip_buttons
    ):

        button.disabled = on

    if on:

        status.value = (
            f"<i style='font:14px system-ui'>"
            f"{esc(text)}"
            f"</i>"
        )

    else:

        status.value = ""


def run_search(text=None):

    text = (
        text
        if text is not None
        else query.value
    ).strip()

    if not text:

        status.value = (
            "<span style='color:#B8352E;"
            "font:14px system-ui'>"
            "Type a description first, or tap a mood."
            "</span>"
        )

        return

    busy(
        True,
        f"Searching {len(df):,} movies..."
    )

    try:

        results = recommend_movies(
            text,
            **current_filters()
        )

        show_results(
            results,
            f"Top {len(results)} for: “{text}”"
        )

        tabs.selected_index = 0

    finally:

        busy(False)




def on_chip(button):

    prompt = dict(
        PROMPT_CHIPS
    )[button.description]

    query.value = prompt

    run_search(prompt)




def on_surprise(_):

    prompt = random.choice(
        SURPRISES
    )

    query.value = prompt

    run_search(prompt)




def on_similar(_):

    index = similar_dd.value

    if index is None:
        return

    title = df.at[
        index,
        title_column
    ]

    busy(
        True,
        f"Finding movies like {title}..."
    )

    try:

        results = recommend_movies(
            query_embedding=
                movie_embeddings[index:index + 1],

            exclude_title=title,

            **current_filters()
        )

        show_results(
            results,
            f"Movies like “{title}”"
        )

        tabs.selected_index = 0

    finally:

        busy(False)




def draw_charts():

    res = state["results"]

    state["charts_stale"] = False

    with charts_out:

        clear_output(
            wait=True
        )

        if res is None or len(res) == 0:

            print(
                "Run a search to see charts."
            )

            return



        fig, ax = plt.subplots(
            figsize=(
                9,
                max(
                    3.5,
                    0.4 * len(res)
                )
            )
        )

        sns.barplot(
            data=res,
            x="similarity_percentage",
            y=title_column,
            color="#2F6F73",
            ax=ax
        )

        ax.set_title(
            "Top recommended movies"
        )

        ax.set_xlabel(
            "Match (%)"
        )

        ax.set_ylabel("")

        plt.tight_layout()
        plt.show()



        rated = res.dropna(
            subset=[rating_column]
        )

        if len(rated) > 0:

            fig, ax = plt.subplots(
                figsize=(9, 4.5)
            )

            sns.scatterplot(
                data=rated,
                x="similarity_percentage",
                y=rating_column,
                s=110,
                color="#F2A93B",
                edgecolor="#17313A",
                ax=ax
            )

            ax.set_title(
                "Rating vs match"
            )

            ax.set_xlabel(
                "Match (%)"
            )

            ax.set_ylabel(
                "Rating"
            )

            plt.tight_layout()
            plt.show()



        gdf = genre_counts(
            res
        ).head(10)

        if len(gdf) > 0:

            fig, ax = plt.subplots(
                figsize=(
                    9,
                    max(
                        3,
                        0.4 * len(gdf)
                    )
                )
            )

            sns.barplot(
                data=gdf,
                x="Count",
                y="Genre",
                color="#B8352E",
                ax=ax
            )

            ax.set_title(
                "Genres in your results"
            )

            ax.set_xlabel(
                "Number of movies"
            )

            ax.set_ylabel("")

            plt.tight_layout()
            plt.show()


def on_tab(change):

    if (
        change["new"] == 1
        and state["charts_stale"]
    ):

        draw_charts()



def on_save(_):

    results = state["results"]

    if results is None or len(results) == 0:
        return

    out_dir = (
        "/content"
        if os.path.isdir("/content")
        else "."
    )

    path = os.path.join(
        out_dir,
        "movie_recommendations.csv"
    )

    results[
        [
            "Rank",
            title_column,
            "similarity_percentage",
            rating_column,
            genre_column
        ]
    ].to_csv(
        path,
        index=False
    )

    try:

        from google.colab import files

        files.download(
            path
        )

        save_msg.value = (
            "<span style='font:14px system-ui'>"
            "Saved and downloading: "
            "movie_recommendations.csv"
            "</span>"
        )

    except Exception:

        save_msg.value = (
            f"<span style='font:14px system-ui'>"
            f"Saved to {esc(path)}"
            f"</span>"
        )


search_btn.on_click(
    lambda _: run_search()
)

surprise_btn.on_click(
    on_surprise
)

for button in chip_buttons:

    button.on_click(
        on_chip
    )

similar_btn.on_click(
    on_similar
)

save_btn.on_click(
    on_save
)

try:

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore"
        )

        query.on_submit(
            lambda _: run_search()
        )

except Exception:

    pass


results_tab = W.VBox([
    status,
    results_html
])


similar_box = W.VBox([

    W.HTML(
        "<b style='font:15px system-ui'>"
        "Loved one of these? "
        "Find more like it"
        "</b>"
    ),

    W.HBox([
        similar_dd,
        similar_btn
    ])

])


tabs = W.Tab([

    W.VBox([
        results_tab,
        similar_box
    ]),

    charts_out,

    W.VBox([
        save_btn,
        save_msg
    ]),

    about

])


for i, title in enumerate([

    "🎟️ Results",
    "📊 Charts",
    "💾 Export",
    "ℹ️ How it works"

]):

    tabs.set_title(
        i,
        title
    )


tabs.observe(
    on_tab,
    names="selected_index"
)


controls = W.VBox([

    W.HBox([
        query,
        search_btn,
        surprise_btn
    ], layout=W.Layout(
        width="100%"
    )),

    W.HBox(
        [
            W.HTML(
                "<span style='font:14px system-ui'>"
                "Quick picks:</span>"
            )
        ] + chip_buttons,

        layout=W.Layout(
            flex_flow="row wrap",
            align_items="center"
        )
    ),

    W.HBox([

        W.VBox([
            n_slider,
            rating_slider
        ]),

        W.VBox([
            genre_dd,
            dedupe
        ]),

        sort_toggle

    ], layout=W.Layout(
        flex_flow="row wrap",
        margin="8px 0 0 0"
    ))

])


app = W.VBox(
    [
        header,
        controls,
        tabs
    ],
    layout=W.Layout(
        max_width="980px"
    )
)


display(app)

print(
    "🎬 Movie Matchmaker is ready!"
)


Loading movie dataset...
Loaded 10,000 movies.
TF-IDF matrix created: (10000, 176)
Loading Sentence-BERT model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence-BERT model loaded successfully.
Embedding cache does not match dataset size.
Creating embeddings for 10,000 movies...
Batch size: 32


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Embeddings saved to: /content/movie_embeddings.npy
Final embedding shape: (10000, 384)
Recommendation engine ready.


🎬 Movie Matchmaker is ready!
